In [ ]:
# Phase 5: UE-Flow-Alert多层异质图构建

**目的**: 构建包含UE、Flow、Alert三层的异质图结构

**输入**: 实体分类和时间窗口数据

**输出**: DGL异质图对象


In [1]:
import dgl
import torch
import pandas as pd
import numpy as np

def construct_heterogeneous_graph(df_window):
    """
    构建UE-Flow-Alert三层异质图
    
    节点类型：
    - UE: 用户设备（基于IP聚合）
    - Flow: 网络流（五元组+时间窗）
    - Alert: 告警（攻击类型+工具）
    
    边类型：
    - UE -> Flow: 'initiates'
    - Flow -> Alert: 'triggers'  
    - Flow -> Flow: 'temporal_adj'
    """
    
    # 构建节点映射
    ue_nodes = df_window[df_window['SrcAddr_Type'] == 'UE']['SrcAddr'].unique()
    flow_nodes = df_window.apply(lambda x: f"{x['SrcAddr']}_{x['DstAddr']}_{x['Proto']}_{x['Sport']}_{x['Dport']}", axis=1).unique()
    alert_nodes = df_window[df_window['Attack Type'] != 'Benign']['Attack Type'].unique()
    
    # 创建节点ID映射
    ue_to_id = {ue: i for i, ue in enumerate(ue_nodes)}
    flow_to_id = {flow: i for i, flow in enumerate(flow_nodes)}
    alert_to_id = {alert: i for i, alert in enumerate(alert_nodes)}
    
    # 构建边
    ue_flow_edges = []
    flow_alert_edges = []
    
    for _, row in df_window.iterrows():
        # UE -> Flow edges
        if row['SrcAddr'] in ue_to_id:
            flow_id = f"{row['SrcAddr']}_{row['DstAddr']}_{row['Proto']}_{row['Sport']}_{row['Dport']}"
            if flow_id in flow_to_id:
                ue_flow_edges.append((ue_to_id[row['SrcAddr']], flow_to_id[flow_id]))
        
        # Flow -> Alert edges
        if row['Attack Type'] != 'Benign':
            flow_id = f"{row['SrcAddr']}_{row['DstAddr']}_{row['Proto']}_{row['Sport']}_{row['Dport']}"
            if flow_id in flow_to_id and row['Attack Type'] in alert_to_id:
                flow_alert_edges.append((flow_to_id[flow_id], alert_to_id[row['Attack Type']]))
    
    # 构建异质图
    graph_data = {
        ('UE', 'initiates', 'Flow'): ue_flow_edges,
        ('Flow', 'triggers', 'Alert'): flow_alert_edges
    }
    
    num_nodes_dict = {
        'UE': len(ue_nodes),
        'Flow': len(flow_nodes), 
        'Alert': len(alert_nodes)
    }
    
    hetero_graph = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)
    
    return hetero_graph, ue_to_id, flow_to_id, alert_to_id

print("异质图构建函数定义完成")


异质图构建函数定义完成


In [ ]:
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm import tqdm

print("Phase 5: UE-Flow-Alert多层异质图构建环境准备完成")

# 加载窗口数据文件列表
fine_window_dir = '../data/processed/windowed_data/fine/'
coarse_window_dir = '../data/processed/windowed_data/coarse/'

fine_files = sorted(glob.glob(os.path.join(fine_window_dir, '*.parquet')))
coarse_files = sorted(glob.glob(os.path.join(coarse_window_dir, '*.parquet')))

print(f"找到细粒度窗口文件: {len(fine_files)} 个")
print(f"找到粗粒度窗口文件: {len(coarse_files)} 个")

# 选择处理的文件子集 (避免内存溢出)
max_files_to_process = 50  # 处理前50个窗口作为示例
fine_files_subset = fine_files[:max_files_to_process]
coarse_files_subset = coarse_files[:max_files_to_process]

print(f"将处理细粒度窗口: {len(fine_files_subset)} 个")
print(f"将处理粗粒度窗口: {len(coarse_files_subset)} 个")


In [ ]:
def extract_flow_features(df_window):
    """从窗口数据中提取流量特征向量"""
    
    # 选择数值型特征 (原始112维特征的子集)
    feature_columns = [
        'Dur', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes',
        'Mean', 'Sum', 'Min', 'Max', 'sTtl', 'dTtl', 'sport', 'dport',
        'traffic_ratio', 'packet_ratio'
    ]
    
    # 添加可用的特征列
    available_features = []
    for col in feature_columns:
        if col in df_window.columns:
            available_features.append(col)
    
    # 提取特征矩阵
    if available_features:
        feature_matrix = df_window[available_features].fillna(0).values
        # 标准化特征
        feature_matrix = (feature_matrix - feature_matrix.mean(axis=0)) / (feature_matrix.std(axis=0) + 1e-8)
    else:
        # 如果没有可用特征，使用基本特征
        feature_matrix = np.random.randn(len(df_window), 16)
    
    return feature_matrix, available_features

def construct_enhanced_heterogeneous_graph(df_window):
    """
    构建增强的UE-Flow-Alert三层异质图
    """
    
    if len(df_window) == 0:
        return None, {}, {}, {}
    
    # 1. 构建节点映射
    # UE节点 (基于源IP地址)
    ue_ips = df_window[df_window['SrcAddr_Type'] == 'UE']['SrcAddr'].unique()
    if len(ue_ips) == 0:
        ue_ips = df_window['SrcAddr'].unique()[:min(10, len(df_window['SrcAddr'].unique()))]
    
    # Flow节点 (基于记录索引)
    flow_indices = df_window.index.tolist()
    
    # Alert节点 (基于攻击类型)
    attack_types = df_window[df_window['Attack Type'] != 'Benign']['Attack Type'].unique()
    if len(attack_types) == 0:
        attack_types = ['Benign']  # 如果没有攻击，添加良性类型
    
    # 创建节点ID映射
    ue_to_id = {ue: i for i, ue in enumerate(ue_ips)}
    flow_to_id = {flow_idx: i for i, flow_idx in enumerate(flow_indices)}
    alert_to_id = {alert: i for i, alert in enumerate(attack_types)}
    
    # 2. 构建边关系
    ue_flow_edges = []
    flow_alert_edges = []
    flow_flow_edges = []  # 时间相邻的流
    
    for idx, row in df_window.iterrows():
        # UE -> Flow edges
        if row['SrcAddr'] in ue_to_id and idx in flow_to_id:
            ue_flow_edges.append((ue_to_id[row['SrcAddr']], flow_to_id[idx]))
        
        # Flow -> Alert edges
        if row['Attack Type'] in alert_to_id and idx in flow_to_id:
            flow_alert_edges.append((flow_to_id[idx], alert_to_id[row['Attack Type']]))
    
    # Flow-Flow temporal edges (基于时间相邻性)
    sorted_flows = df_window.sort_values('StartTime_unix')
    for i in range(len(sorted_flows) - 1):
        current_idx = sorted_flows.iloc[i].name
        next_idx = sorted_flows.iloc[i + 1].name
        if current_idx in flow_to_id and next_idx in flow_to_id:
            flow_flow_edges.append((flow_to_id[current_idx], flow_to_id[next_idx]))
    
    # 3. 构建异质图
    graph_data = {}
    
    # 添加边类型 (确保至少有一个边)
    if ue_flow_edges:
        graph_data[('UE', 'initiates', 'Flow')] = ue_flow_edges
    if flow_alert_edges:
        graph_data[('Flow', 'triggers', 'Alert')] = flow_alert_edges
    if flow_flow_edges:
        graph_data[('Flow', 'temporal_adj', 'Flow')] = flow_flow_edges
    
    # 如果没有边，创建自环
    if not graph_data:
        if len(flow_indices) > 0:
            graph_data[('Flow', 'self_loop', 'Flow')] = [(0, 0)]
        else:
            return None, {}, {}, {}
    
    num_nodes_dict = {
        'UE': len(ue_ips),
        'Flow': len(flow_indices),
        'Alert': len(attack_types)
    }
    
    try:
        hetero_graph = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)
        
        # 4. 添加节点特征
        flow_features, feature_names = extract_flow_features(df_window)
        
        # Flow节点特征
        hetero_graph.nodes['Flow'].data['feat'] = torch.FloatTensor(flow_features)
        
        # UE节点特征 (简化特征)
        ue_features = np.random.randn(len(ue_ips), 16)  # 16维UE特征
        hetero_graph.nodes['UE'].data['feat'] = torch.FloatTensor(ue_features)
        
        # Alert节点特征 (基于攻击类型的one-hot编码)
        alert_features = np.eye(len(attack_types))
        hetero_graph.nodes['Alert'].data['feat'] = torch.FloatTensor(alert_features)
        
        return hetero_graph, ue_to_id, flow_to_id, alert_to_id
        
    except Exception as e:
        print(f"构建图时出错: {e}")
        return None, {}, {}, {}

print("增强的异质图构建函数定义完成")


In [ ]:
# 批量处理细粒度窗口并构建图
print("\\n开始批量构建细粒度异质图...")

fine_graphs = []
fine_graph_stats = []
successful_fine_graphs = 0

for i, file_path in enumerate(tqdm(fine_files_subset, desc="构建细粒度图")):
    try:
        # 加载窗口数据
        df_window = pd.read_parquet(file_path)
        
        # 构建异质图
        graph, ue_mapping, flow_mapping, alert_mapping = construct_enhanced_heterogeneous_graph(df_window)
        
        if graph is not None:
            # 保存图对象
            timestamp = df_window['window_start'].iloc[0] if 'window_start' in df_window.columns else i
            graph_filename = f'hetero_graph_fine_t{timestamp:.0f}.dgl'
            graph_path = os.path.join('../data/processed/graphs/', graph_filename)
            dgl.save_graphs(graph_path, [graph])
            
            # 收集统计信息
            stats = {
                'window_id': i,
                'timestamp': timestamp,
                'num_ue_nodes': graph.num_nodes('UE'),
                'num_flow_nodes': graph.num_nodes('Flow'),
                'num_alert_nodes': graph.num_nodes('Alert'),
                'num_edges': graph.num_edges(),
                'num_records': len(df_window),
                'attack_types': df_window['Attack Type'].unique().tolist(),
                'graph_file': graph_filename
            }
            
            fine_graph_stats.append(stats)
            successful_fine_graphs += 1
            
        else:
            print(f"跳过空窗口: {file_path}")
            
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")
        continue

print(f"成功构建细粒度图: {successful_fine_graphs}/{len(fine_files_subset)}")

# 批量处理粗粒度窗口
print("\\n开始批量构建粗粒度异质图...")

coarse_graphs = []
coarse_graph_stats = []
successful_coarse_graphs = 0

for i, file_path in enumerate(tqdm(coarse_files_subset, desc="构建粗粒度图")):
    try:
        # 加载窗口数据
        df_window = pd.read_parquet(file_path)
        
        # 构建异质图
        graph, ue_mapping, flow_mapping, alert_mapping = construct_enhanced_heterogeneous_graph(df_window)
        
        if graph is not None:
            # 保存图对象
            timestamp = df_window['window_start'].iloc[0] if 'window_start' in df_window.columns else i
            graph_filename = f'hetero_graph_coarse_t{timestamp:.0f}.dgl'
            graph_path = os.path.join('../data/processed/graphs/', graph_filename)
            dgl.save_graphs(graph_path, [graph])
            
            # 收集统计信息
            stats = {
                'window_id': i,
                'timestamp': timestamp,
                'num_ue_nodes': graph.num_nodes('UE'),
                'num_flow_nodes': graph.num_nodes('Flow'),
                'num_alert_nodes': graph.num_nodes('Alert'),
                'num_edges': graph.num_edges(),
                'num_records': len(df_window),
                'attack_types': df_window['Attack Type'].unique().tolist(),
                'graph_file': graph_filename
            }
            
            coarse_graph_stats.append(stats)
            successful_coarse_graphs += 1
            
        else:
            print(f"跳过空窗口: {file_path}")
            
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")
        continue

print(f"成功构建粗粒度图: {successful_coarse_graphs}/{len(coarse_files_subset)}")


In [ ]:
# 生成图统计报告
def generate_graph_statistics(fine_stats, coarse_stats):
    """生成图统计信息"""
    
    overall_stats = {
        "timestamp": datetime.now().isoformat(),
        "fine_graphs": {
            "total_graphs": len(fine_stats),
            "successful_graphs": successful_fine_graphs,
            "avg_ue_nodes": np.mean([s['num_ue_nodes'] for s in fine_stats]) if fine_stats else 0,
            "avg_flow_nodes": np.mean([s['num_flow_nodes'] for s in fine_stats]) if fine_stats else 0,
            "avg_alert_nodes": np.mean([s['num_alert_nodes'] for s in fine_stats]) if fine_stats else 0,
            "avg_edges": np.mean([s['num_edges'] for s in fine_stats]) if fine_stats else 0,
            "avg_records": np.mean([s['num_records'] for s in fine_stats]) if fine_stats else 0
        },
        "coarse_graphs": {
            "total_graphs": len(coarse_stats),
            "successful_graphs": successful_coarse_graphs,
            "avg_ue_nodes": np.mean([s['num_ue_nodes'] for s in coarse_stats]) if coarse_stats else 0,
            "avg_flow_nodes": np.mean([s['num_flow_nodes'] for s in coarse_stats]) if coarse_stats else 0,
            "avg_alert_nodes": np.mean([s['num_alert_nodes'] for s in coarse_stats]) if coarse_stats else 0,
            "avg_edges": np.mean([s['num_edges'] for s in coarse_stats]) if coarse_stats else 0,
            "avg_records": np.mean([s['num_records'] for s in coarse_stats]) if coarse_stats else 0
        },
        "detailed_stats": {
            "fine_graph_details": fine_stats,
            "coarse_graph_details": coarse_stats
        }
    }
    
    return overall_stats

# 生成统计报告
graph_statistics = generate_graph_statistics(fine_graph_stats, coarse_graph_stats)

print("\\n=== 图构建统计摘要 ===")
print(f"细粒度图: {graph_statistics['fine_graphs']['successful_graphs']}/{graph_statistics['fine_graphs']['total_graphs']}")
print(f"  - 平均UE节点数: {graph_statistics['fine_graphs']['avg_ue_nodes']:.2f}")
print(f"  - 平均Flow节点数: {graph_statistics['fine_graphs']['avg_flow_nodes']:.2f}")
print(f"  - 平均Alert节点数: {graph_statistics['fine_graphs']['avg_alert_nodes']:.2f}")
print(f"  - 平均边数: {graph_statistics['fine_graphs']['avg_edges']:.2f}")

print(f"\\n粗粒度图: {graph_statistics['coarse_graphs']['successful_graphs']}/{graph_statistics['coarse_graphs']['total_graphs']}")
print(f"  - 平均UE节点数: {graph_statistics['coarse_graphs']['avg_ue_nodes']:.2f}")
print(f"  - 平均Flow节点数: {graph_statistics['coarse_graphs']['avg_flow_nodes']:.2f}")
print(f"  - 平均Alert节点数: {graph_statistics['coarse_graphs']['avg_alert_nodes']:.2f}")
print(f"  - 平均边数: {graph_statistics['coarse_graphs']['avg_edges']:.2f}")

# 可视化图统计信息
plt.figure(figsize=(16, 12))

# 1. 节点数量分布 (细粒度)
plt.subplot(2, 3, 1)
if fine_graph_stats:
    ue_nodes = [s['num_ue_nodes'] for s in fine_graph_stats]
    flow_nodes = [s['num_flow_nodes'] for s in fine_graph_stats]
    alert_nodes = [s['num_alert_nodes'] for s in fine_graph_stats]
    
    x = np.arange(len(fine_graph_stats))
    width = 0.25
    
    plt.bar(x - width, ue_nodes, width, label='UE', alpha=0.7)
    plt.bar(x, flow_nodes, width, label='Flow', alpha=0.7)
    plt.bar(x + width, alert_nodes, width, label='Alert', alpha=0.7)
    
    plt.title('细粒度图节点数量分布')
    plt.xlabel('图ID')
    plt.ylabel('节点数')
    plt.legend()

# 2. 节点数量分布 (粗粒度)
plt.subplot(2, 3, 2)
if coarse_graph_stats:
    ue_nodes = [s['num_ue_nodes'] for s in coarse_graph_stats]
    flow_nodes = [s['num_flow_nodes'] for s in coarse_graph_stats]
    alert_nodes = [s['num_alert_nodes'] for s in coarse_graph_stats]
    
    x = np.arange(len(coarse_graph_stats))
    width = 0.25
    
    plt.bar(x - width, ue_nodes, width, label='UE', alpha=0.7)
    plt.bar(x, flow_nodes, width, label='Flow', alpha=0.7)
    plt.bar(x + width, alert_nodes, width, label='Alert', alpha=0.7)
    
    plt.title('粗粒度图节点数量分布')
    plt.xlabel('图ID')
    plt.ylabel('节点数')
    plt.legend()

# 3. 边数量分布对比
plt.subplot(2, 3, 3)
if fine_graph_stats and coarse_graph_stats:
    fine_edges = [s['num_edges'] for s in fine_graph_stats]
    coarse_edges = [s['num_edges'] for s in coarse_graph_stats]
    
    plt.hist(fine_edges, bins=20, alpha=0.7, label='细粒度', color='blue')
    plt.hist(coarse_edges, bins=20, alpha=0.7, label='粗粒度', color='red')
    plt.title('边数量分布对比')
    plt.xlabel('边数')
    plt.ylabel('频率')
    plt.legend()

# 4. 图规模随时间变化 (细粒度)
plt.subplot(2, 3, 4)
if fine_graph_stats:
    timestamps = [s['timestamp'] for s in fine_graph_stats]
    edge_counts = [s['num_edges'] for s in fine_graph_stats]
    plt.plot(edge_counts, marker='o', alpha=0.7)
    plt.title('细粒度图规模时间序列')
    plt.xlabel('时间窗口')
    plt.ylabel('边数')

# 5. 图规模随时间变化 (粗粒度)
plt.subplot(2, 3, 5)
if coarse_graph_stats:
    timestamps = [s['timestamp'] for s in coarse_graph_stats]
    edge_counts = [s['num_edges'] for s in coarse_graph_stats]
    plt.plot(edge_counts, marker='s', alpha=0.7, color='red')
    plt.title('粗粒度图规模时间序列')
    plt.xlabel('时间窗口')
    plt.ylabel('边数')

# 6. 攻击类型频率
plt.subplot(2, 3, 6)
all_attack_types = []
for stats in fine_graph_stats + coarse_graph_stats:
    all_attack_types.extend(stats['attack_types'])

if all_attack_types:
    from collections import Counter
    attack_counts = Counter(all_attack_types)
    attack_names = list(attack_counts.keys())
    attack_frequencies = list(attack_counts.values())
    
    plt.bar(range(len(attack_names)), attack_frequencies)
    plt.title('攻击类型频率分布')
    plt.xlabel('攻击类型')
    plt.ylabel('出现次数')
    plt.xticks(range(len(attack_names)), attack_names, rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/metadata/graph_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

# 保存统计报告
with open('../data/processed/metadata/graph_statistics.json', 'w', encoding='utf-8') as f:
    json.dump(graph_statistics, f, indent=2, ensure_ascii=False)

print("\\n✓ Phase 5 完成:")
print(f"  - 细粒度图文件: {successful_fine_graphs} 个")
print(f"  - 粗粒度图文件: {successful_coarse_graphs} 个")
print(f"  - 图统计报告: ../data/processed/metadata/graph_statistics.json")
print(f"  - 图统计可视化: ../data/processed/metadata/graph_statistics.png")
print(f"\\n图数据保存在: ../data/processed/graphs/")
print(f"总计图文件: {successful_fine_graphs + successful_coarse_graphs} 个")
